# Lesson 19 — Event-Driven Architecture

## What you will learn
- **Event-driven vs request-driven** agent triggering
- How to trigger a LangGraph agent from an **event queue** (SQS-style)
- **Webhook receiver** that invokes a graph per incoming event
- **Background worker** pattern: poll queue → process → acknowledge
- **Dead-letter queue (DLQ)** simulation for failed events

## Architecture
```
External system (DB, S3, webhook)
    ↓  event
Event Queue  (SQS / in-memory queue)
    ↓  worker polls
Worker loop → graph.invoke(event_as_state)
    ↓  success: ack message
    ↓  failure: move to DLQ after 3 retries
```

In [ ]:
# !pip install langgraph langchain-ollama

## Step 1 — In-Memory Event Queue (SQS simulation)

In [ ]:
import queue
import threading
import time
import uuid
from dataclasses import dataclass, field
from typing import Annotated, Any
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

@dataclass
class Event:
    event_id:   str
    event_type: str          # "user_message", "file_uploaded", "scheduled"
    payload:    dict
    attempt:    int = 0
    max_retries: int = 3

# Queues
event_queue = queue.Queue()
dlq         = queue.Queue()   # dead-letter queue
results     = []

def enqueue(event_type: str, payload: dict) -> Event:
    event = Event(
        event_id=str(uuid.uuid4())[:8],
        event_type=event_type,
        payload=payload,
    )
    event_queue.put(event)
    print(f"  [queue] Enqueued {event.event_id}: {event_type}")
    return event

print("Event queue ready")

## Step 2 — Event Processing Graph

In [ ]:
llm = ChatOllama(model="llama3", temperature=0)

class EventState(TypedDict):
    messages:   Annotated[list, add_messages]
    event_id:   str
    event_type: str
    payload:    dict
    result:     str
    error:      str

def route_event(state: EventState) -> str:
    """Route to different handlers based on event type."""
    return state["event_type"]

def handle_user_message(state: EventState) -> dict:
    msg = state["payload"].get("message", "")
    response = llm.invoke([SystemMessage(content="Reply briefly."), HumanMessage(content=msg)])
    return {"result": response.content, "messages": [response]}

def handle_file_uploaded(state: EventState) -> dict:
    filename = state["payload"].get("filename", "unknown")
    return {"result": f"File '{filename}' processed and indexed."}

def handle_scheduled(state: EventState) -> dict:
    task = state["payload"].get("task", "")
    return {"result": f"Scheduled task '{task}' completed at {time.strftime('%H:%M:%S')}."}

def handle_unknown(state: EventState) -> dict:
    return {"error": f"Unknown event type: {state['event_type']}", "result": ""}

builder = StateGraph(EventState)
builder.add_node("route",           lambda s: {})   # passthrough
builder.add_node("user_message",    handle_user_message)
builder.add_node("file_uploaded",   handle_file_uploaded)
builder.add_node("scheduled",       handle_scheduled)
builder.add_node("unknown",         handle_unknown)

builder.add_edge(START, "route")
builder.add_conditional_edges("route", route_event, {
    "user_message":  "user_message",
    "file_uploaded": "file_uploaded",
    "scheduled":     "scheduled",
})
# Default to unknown for unregistered types
builder.add_edge("user_message",  END)
builder.add_edge("file_uploaded", END)
builder.add_edge("scheduled",     END)
builder.add_edge("unknown",       END)

graph = builder.compile(checkpointer=MemorySaver())
print("Event-driven graph compiled!")

## Step 3 — Worker Loop with Retry + DLQ

In [ ]:
def process_event(event: Event) -> bool:
    """Returns True on success, False on failure."""
    try:
        config = {"configurable": {"thread_id": event.event_id}}
        result = graph.invoke({
            "messages":   [],
            "event_id":   event.event_id,
            "event_type": event.event_type,
            "payload":    event.payload,
            "result":     "",
            "error":      "",
        }, config=config)
        
        if result.get("error"):
            print(f"  [worker] Event {event.event_id} processing error: {result['error']}")
            return False
        
        results.append({"event_id": event.event_id, "result": result.get("result", "")})
        print(f"  [worker] ✅ {event.event_id} ({event.event_type}): {result.get('result', '')[:60]}")
        return True
    except Exception as e:
        print(f"  [worker] ❌ {event.event_id} error: {e}")
        return False

def worker_loop(timeout: float = 5.0):
    """Poll queue, process events, retry failures, send to DLQ after max_retries."""
    print("[worker] Starting...")
    deadline = time.time() + timeout
    
    while time.time() < deadline:
        try:
            event = event_queue.get(timeout=0.5)
        except queue.Empty:
            continue
        
        success = process_event(event)
        if not success:
            event.attempt += 1
            if event.attempt < event.max_retries:
                print(f"  [worker] Retrying {event.event_id} (attempt {event.attempt}/{event.max_retries})")
                event_queue.put(event)
            else:
                print(f"  [worker] DLQ: {event.event_id} failed after {event.max_retries} attempts")
                dlq.put(event)
        
        event_queue.task_done()
    
    print("[worker] Done.")

print("Worker function ready")

## Step 4 — Publish Events and Run Worker

In [ ]:
# Publish a batch of events
enqueue("user_message",  {"message": "What is LangGraph?", "user_id": "alice"})
enqueue("file_uploaded", {"filename": "report_q1.pdf", "size_bytes": 204800})
enqueue("scheduled",     {"task": "weekly_summary", "cron": "0 9 * * 1"})
enqueue("user_message",  {"message": "How does HITL work?", "user_id": "bob"})

print(f"\nQueue size: {event_queue.qsize()} events")
print("\nRunning worker...")

# Run worker (blocks until queue is empty or timeout)
worker_loop(timeout=60.0)

print(f"\nResults processed: {len(results)}")
print(f"DLQ size: {dlq.qsize()}")

## Key Takeaways

| Pattern | Implementation |
|---------|---------------|
| Event queue | `queue.Queue` (production: AWS SQS, Redis Streams) |
| Event routing | `event_type` in state → `add_conditional_edges` |
| Retry logic | Re-enqueue on failure, track `attempt` counter |
| Dead-letter queue | Move to DLQ after `max_retries` exceeded |
| Worker loop | Poll → process → ack; runs as background thread or Lambda |

## 🏋️ Exercise
1. Add a `priority` field to `Event` — process `priority=1` events before `priority=2`
2. Add a `batch_process(events: list)` function that invokes the graph for multiple events using `asyncio.gather()`
3. Simulate a failing event: raise an exception in `handle_file_uploaded` 2 out of 3 times (random) — verify it ends up in DLQ after 3 retries

In [ ]:
# Your exercise solution here
